# Train Cross-Encoder NLI tren Kaggle


## 1. Cai thu vien

In [1]:
%pip install -q "transformers==4.45.2" "datasets>=3.0" "accelerate>=1.0" "evaluate>=0.4" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=15.0" "huggingface_hub>=0.23"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requir

## 2. Tai source code tu GitHub

In [2]:
# Dien URL repo GitHub cua nhom vao day truoc khi chay tren Kaggle.
GITHUB_REPOSITORY_URL = "https://github.com/PhDQuang/similarity_search.git"  # vi du: "https://github.com/<user>/<repo>.git"

if GITHUB_REPOSITORY_URL:
    !rm -rf /kaggle/working/similarity_search
    !git clone {GITHUB_REPOSITORY_URL} /kaggle/working/similarity_search
    %cd /kaggle/working/similarity_search
else:
    print("Hay dien GITHUB_REPOSITORY_URL, hoac upload repo source thanh Kaggle Dataset roi cd vao thu muc do.")

Cloning into '/kaggle/working/similarity_search'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 86 (delta 18), reused 79 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 349.59 KiB | 4.16 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/kaggle/working/similarity_search


## 3. Cai package local cua project

In [3]:
!python -m pip install -q -e .
!python -m similarity_search.models.train_cross_encoder --help

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for similarity-search-nlp (pyproject.toml) ... done
2026-06-16 09:20:24.220295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781601624.452075      79 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781601624.526983      79 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781601625.086384      79 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same targe

## 4. Kiem tra GPU

In [4]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: True
Tesla T4


## 5. Smoke test nhanh

Chay truoc voi it mau de kiem tra pipeline. Neu thanh cong moi chay full training.

In [5]:
!python -m similarity_search.models.train_cross_encoder \
  --output-dir /kaggle/working/cross-encoder-smoke \
  --result-dir /kaggle/working/cross-encoder-smoke-outputs \
  --max-train-samples 2000 \
  --max-dev-samples 500 \
  --max-test-samples 500 \
  --num-train-epochs 0.05 \
  --batch-size 16 \
  --eval-batch-size 32 \
  --eval-steps 20 \
  --save-steps 20 \
  --logging-steps 10 \
  --rerank-queries 50 \
  --rerank-pool-size 10 \
  --skip-benchmark

2026-06-16 09:20:58.914133: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781601658.939745      99 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781601658.948391      99 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781601658.969902      99 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781601658.969932      99 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781601658.969936      99 computation_placer.cc:177] computation placer alr

## 6. Full training Cross-Encoder NLI

Cau hinh mac dinh dung `distilbert-base-uncased`, train 300k mau AllNLI `pair-class`, 1 epoch. Neu bi OOM, giam `--batch-size 16` va tang `--gradient-accumulation-steps 2`.

In [6]:
!python -m similarity_search.models.train_cross_encoder \
  --train-dataset-name sentence-transformers/all-nli \
  --train-dataset-config pair-class \
  --benchmark-dataset-name phdquang/allnli-pair-class-processed \
  --base-model distilbert-base-uncased \
  --output-dir /kaggle/working/allnli-cross-encoder-nli \
  --result-dir /kaggle/working/cross_encoder_outputs \
  --max-train-samples 300000 \
  --max-dev-samples 20000 \
  --max-test-samples 20000 \
  --num-train-epochs 1 \
  --batch-size 32 \
  --eval-batch-size 64 \
  --gradient-accumulation-steps 1 \
  --learning-rate 2e-5 \
  --warmup-ratio 0.1 \
  --weight-decay 0.01 \
  --max-length 128 \
  --eval-steps 1000 \
  --save-steps 1000 \
  --logging-steps 100 \
  --rerank-queries 1000 \
  --rerank-pool-size 20

2026-06-16 09:22:30.075482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781601750.102321     252 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781601750.111471     252 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781601750.133920     252 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781601750.133981     252 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781601750.133986     252 computation_placer.cc:177] computation placer alr

In [7]:
# from kaggle_secrets import UserSecretsClient
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
#
# !python -m similarity_search.models.train_cross_encoder \
#   --train-dataset-name sentence-transformers/all-nli \
#   --train-dataset-config pair-class \
#   --base-model distilbert-base-uncased \
#   --output-dir /kaggle/working/allnli-cross-encoder-nli-hub \
#   --result-dir /kaggle/working/cross_encoder_outputs_hub \
#   --max-train-samples 300000 \
#   --num-train-epochs 1 \
#   --batch-size 32 \
#   --eval-batch-size 64 \
#   --learning-rate 2e-5 \
#   --push-to-hub \
#   --hub-model-id <username-or-team>/allnli-distilbert-cross-encoder-nli \
#   --hub-private-repo